## Bước 1 — Cấu hình và kiểm tra

In [ ]:
import os, sys, json, shutil, subprocess

VOL        = "/mnt/m3diff"                       # Volume tạo ở Files panel
REPO       = "/root/M3Diff"
VENV       = "/root/m3diff-venv"
PY         = f"{VENV}/bin/python"
CKPT       = "/root/ckpt/M3Diff"                 # checkpoint 16GB, tải lại mỗi phiên (nhanh) 
RESULTS    = f"{VOL}/results/OmniDiff_Real"

# A10 / L4 / A100 (>= 24GB): QUANT="none", DTYPE="bfloat16"  -> đúng bản gốc
# T4 (16GB):                 QUANT="8bit", DTYPE="float16"
QUANT = "none"
DTYPE = "bfloat16"

# Chỉ cần nếu tải Hugging Face báo lỗi 401/403: dán token vào đây
HF_TOKEN = os.environ.get("HF_TOKEN", "")

!nvidia-smi --query-gpu=name,memory.total --format=csv
if not os.path.isdir(VOL):
    print(f"\n!!! Chưa thấy Volume {VOL}. Vào Files -> tạo Volume tên m3diff. Hiện /mnt có:", os.listdir("/mnt") if os.path.isdir("/mnt") else "(trống)")
else:
    os.makedirs(RESULTS, exist_ok=True)
    print("\nVolume OK:", VOL)

## Bước 2 — Tạo môi trường Python 3.10 riêng + cài thư viện
Kernel của Modal là Python 3.12 với torch bản mới. Repo cần đúng torch 2.2.2 / transformers 4.40.0, nên tạo venv riêng bằng `uv` (không đụng vào kernel, không phải restart).

In [ ]:
%uv pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"

## Bước 3 — Tải checkpoint M³Diff (16GB) và ảnh test OmniDiff-Real
Chỉ tải phần ảnh `Real/test` (bộ test cần), không tải cả dataset.

In [ ]:
dl = f'''
from huggingface_hub import snapshot_download
tok = {HF_TOKEN!r} or None
snapshot_download("IVC-liuyuan/M3Diff", local_dir="{CKPT}", token=tok)
snapshot_download("IVC-liuyuan/OmniDiff", repo_type="dataset", local_dir="{REPO}/datasets/OmniDiff",
                  allow_patterns=["Real/test/*"], token=tok)
print("Tải xong")
'''
subprocess.run([PY, "-c", dl], check=True)

with open(f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test.json") as f:
    questions = json.load(f)
needed = sorted({img for q in questions for img in q["image"]})
missing = [p for p in needed if not os.path.exists(f"{REPO}/datasets/OmniDiff/{p}")]
!du -sh {CKPT}
print(f"Ảnh cần: {len(needed)} | còn thiếu: {len(missing)}", missing[:10])

## Bước 4 — Patch script eval
1. Đặt toàn bộ model lên GPU (tránh lỗi `Cannot copy out of meta tensor`); nếu `QUANT` là 8bit/4bit thì nén phần LLM.
2. Chạy tiếp được: bỏ qua câu đã có trong file kết quả, ghi file sau từng câu.

Chạy lại nhiều lần không sao.

In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

## Bước 5a — Ghi script căn chỉnh (chỉ ghi file, chạy nhanh)
Cho mỗi cặp: tìm điểm đặc trưng SIFT → ghép theo tỉ lệ Lowe 0,75 → homography bằng RANSAC.
Chỉ căn khi đáng tin (≥ 30 điểm khớp đúng, tỉ lệ khớp đúng ≥ 30%, biến đổi không bất thường) và lệch ≥ 0,3% đường chéo; còn lại giữ nguyên ảnh gốc.
Viền trống sau khi warp được lấp bằng điểm ảnh của ảnh trước để model không tưởng viền đen là thay đổi.

In [ ]:
%%writefile /root/align_pairs.py
"""Căn chỉnh ảnh sau (B) theo ảnh trước (A) cho từng cặp: SIFT + homography (RANSAC).

- Chỉ căn khi ghép điểm đáng tin; nếu không, giữ nguyên ảnh gốc.
- Phần viền không có dữ liệu sau khi warp được lấp bằng điểm ảnh của A
  (để model không tưởng viền đen là thay đổi).
- Ghi ra file câu hỏi mới trỏ tới ảnh B đã căn, dùng thẳng cho script eval.
"""
import argparse, json, os

import cv2
import numpy as np
import pandas as pd

MATCH_SIDE = 1024      # cạnh dài khi tìm điểm đặc trưng
MIN_INLIERS = 30
MIN_INLIER_RATIO = 0.30
MIN_DISP_PCT = 0.3     # lệch dưới mức này coi như đã thẳng hàng, không căn


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--questions", required=True)
    p.add_argument("--image-root", required=True)
    p.add_argument("--out-subdir", required=True, help="thư mục con trong image-root để lưu ảnh B đã căn")
    p.add_argument("--out-questions", required=True)
    p.add_argument("--stats", required=True)
    p.add_argument("--examples-dir", default=None)
    return p.parse_args()


def resize_max(img, side):
    s = min(1.0, side / max(img.shape[:2]))
    return (cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img), s


def estimate_homography(A, B):
    """Trả về (H ánh xạ toạ độ B -> A ở độ phân giải gốc, thông tin) hoặc (None, thông tin)."""
    a, sa = resize_max(A, MATCH_SIDE)
    b, sb = resize_max(B, MATCH_SIDE)
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=4000)
    ka, da = sift.detectAndCompute(ga, None)
    kb, db = sift.detectAndCompute(gb, None)
    info = {"inliers": 0, "inlier_ratio": 0.0, "good_matches": 0}
    if da is None or db is None or len(ka) < 20 or len(kb) < 20:
        return None, info | {"reason": "ít điểm đặc trưng"}
    knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(db, da, k=2)
    good = [m for m, n in (p for p in knn if len(p) == 2) if m.distance < 0.75 * n.distance]
    info["good_matches"] = len(good)
    if len(good) < MIN_INLIERS:
        return None, info | {"reason": "ít cặp điểm khớp"}
    pb = np.float32([kb[m.queryIdx].pt for m in good]) / sb
    pa = np.float32([ka[m.trainIdx].pt for m in good]) / sa
    H, mask = cv2.findHomography(pb, pa, cv2.RANSAC, ransacReprojThreshold=3.0 / sa)
    if H is None:
        return None, info | {"reason": "không ước lượng được homography"}
    inl = int(mask.sum())
    info |= {"inliers": inl, "inlier_ratio": inl / len(good)}
    if inl < MIN_INLIERS or inl / len(good) < MIN_INLIER_RATIO:
        return None, info | {"reason": "ít điểm khớp đúng"}
    det = np.linalg.det(H[:2, :2])
    if not 0.5 < det < 2.0:
        return None, info | {"reason": "biến đổi bất thường (tỉ lệ)"}
    return H, info


def corner_displacement_pct(H, B_shape, A_shape):
    h, w = B_shape[:2]
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    moved = cv2.perspectiveTransform(corners, H).reshape(-1, 2)
    diag = float(np.hypot(*A_shape[:2]))
    return float(np.linalg.norm(moved - corners.reshape(-1, 2), axis=1).mean() / diag * 100), moved


def warp_with_fill(A, B, H):
    h, w = A.shape[:2]
    Bw = cv2.warpPerspective(B, H, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    valid = cv2.warpPerspective(np.full(B.shape[:2], 255, np.uint8), H, (w, h), flags=cv2.INTER_NEAREST) > 0
    out = np.where(valid[..., None], Bw, A)
    return out, float(valid.mean() * 100)


def diff_gray(X, Y):
    x, _ = resize_max(X, 512)
    y = cv2.resize(Y, (x.shape[1], x.shape[0]), interpolation=cv2.INTER_AREA)
    return cv2.absdiff(cv2.cvtColor(x, cv2.COLOR_BGR2GRAY), cv2.cvtColor(y, cv2.COLOR_BGR2GRAY))


def save_example(path, qid, A, B, Ba, disp):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    panels = [(A, "Ảnh trước (A)"), (B, "Ảnh sau gốc (B)"), (Ba, "Ảnh sau đã căn theo A"),
              (diff_gray(A, B), "|A − B| trước khi căn"), (diff_gray(A, Ba), "|A − B| sau khi căn")]
    fig, axes = plt.subplots(1, 5, figsize=(22, 4.6), facecolor="#fcfcfb")
    for ax, (im, t) in zip(axes, panels):
        if im.ndim == 3:
            ax.imshow(cv2.cvtColor(resize_max(im, 512)[0], cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap="gray", vmin=0, vmax=255)
        ax.set_title(t, fontsize=10, color="#0b0b0b", loc="left"); ax.axis("off")
    fig.suptitle(f"{qid} — độ lệch trước khi căn ≈ {disp:.1f}% đường chéo ảnh", x=0.01, ha="left", fontsize=11, color="#0b0b0b")
    fig.tight_layout()
    fig.savefig(path, dpi=110, facecolor="#fcfcfb")
    plt.close(fig)


def main():
    args = parse()
    questions = json.load(open(args.questions))
    out_dir = os.path.join(args.image_root, args.out_subdir)
    os.makedirs(out_dir, exist_ok=True)
    if args.examples_dir:
        os.makedirs(args.examples_dir, exist_ok=True)
    cache, rows, new_q = {}, [], []

    def load(rel):
        if rel not in cache:
            img = cv2.imread(os.path.join(args.image_root, rel), cv2.IMREAD_COLOR)
            if img is None:
                raise FileNotFoundError(rel)
            cache[rel] = img
        return cache[rel]

    for k, q in enumerate(questions):
        ra, rb = q["image"]
        A, B = load(ra), load(rb)
        H, info = estimate_homography(A, B)
        row = {"id": q["id"], "image_a": ra, "image_b": rb} | {k2: v for k2, v in info.items()}
        new_b = rb
        if H is None:
            row["status"] = "không căn (ghép thất bại)"
        else:
            disp, _ = corner_displacement_pct(H, B.shape, A.shape)
            row["disp_pct"] = disp
            if disp < MIN_DISP_PCT:
                row["status"] = "không căn (đã thẳng hàng)"
            else:
                Ba, valid_pct = warp_with_fill(A, B, H)
                new_b = f"{args.out_subdir}/{q['id']}_B.jpg"
                cv2.imwrite(os.path.join(args.image_root, new_b), Ba, [cv2.IMWRITE_JPEG_QUALITY, 95])
                row |= {"status": "đã căn", "valid_area_pct": valid_pct}
        row["new_image_b"] = new_b
        rows.append(row)
        new_q.append(q | {"image": [ra, new_b]})
        if (k + 1) % 100 == 0:
            print(f"  {k+1}/{len(questions)}", flush=True)
        if len(cache) > 40:
            cache.clear()

    json.dump(new_q, open(args.out_questions, "w"), indent=2)
    df = pd.DataFrame(rows)
    df.to_csv(args.stats, index=False)

    if args.examples_dir:
        top = df[df["status"] == "đã căn"].nlargest(3, "disp_pct")
        for _, r in top.iterrows():
            A, B = load(r["image_a"]), load(r["image_b"])
            Ba = cv2.imread(os.path.join(args.image_root, r["new_image_b"]))
            save_example(os.path.join(args.examples_dir, f"align_{r['id']}.png"), r["id"], A, B, Ba, r["disp_pct"])

    print("\nKết quả căn chỉnh:")
    print(df["status"].value_counts().to_string())
    al = df[df["status"] == "đã căn"]
    if len(al):
        print(f"Độ lệch của các cặp đã căn: trung vị {al['disp_pct'].median():.2f}%, lớn nhất {al['disp_pct'].max():.1f}% đường chéo")
        print(f"Phần ảnh B còn dữ liệu sau khi warp: trung vị {al['valid_area_pct'].median():.1f}%")


if __name__ == "__main__":
    main()

## Bước 5b — Ghi script so sánh (chỉ ghi file, chạy nhanh)

In [ ]:
%%writefile /root/compare_runs.py
"""So sánh 2 lần chạy (baseline và ảnh đã căn): điểm toàn bộ, điểm trên nhóm cặp đã căn, kiểm định Wilcoxon."""
import argparse, json, os, sys

VALID_MIN = 90.0   # chốt trước khi xem kết quả: chỉ dùng ảnh đã căn khi viền phải lấp từ A < 10% diện tích

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--repo", required=True)
    p.add_argument("--reference", required=True)
    p.add_argument("--baseline", required=True)
    p.add_argument("--aligned", required=True)
    p.add_argument("--stats", required=True, help="stats.csv của bước căn chỉnh")
    p.add_argument("--out", required=True)
    return p.parse_args()


def score(repo, reference, predictions):
    sys.path.insert(0, repo)
    from pathlib import Path
    from metrics import load_predictions
    from pycocotools.coco import COCO
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider

    coco = COCO(reference)
    res = coco.loadRes(load_predictions(Path(predictions)))
    ids = res.getImgIds()
    tok = PTBTokenizer()
    gts = tok.tokenize({i: coco.imgToAnns[i] for i in ids})
    rs = tok.tokenize({i: res.imgToAnns[i] for i in ids})
    keys = list(gts.keys())
    corpus, per = {}, pd.DataFrame({"id": keys})
    for scorer, names in [(Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]), (Meteor(), ["METEOR"]),
                          (Rouge(), ["ROUGE_L"]), (Cider(), ["CIDEr"])]:
        s, per_s = scorer.compute_score(gts, rs)
        s = s if isinstance(s, list) else [s]
        per_s = per_s if names[0] != "Bleu_1" else per_s[3]    # BLEU-4 từng câu
        for n, v in zip(names, s):
            corpus[n] = float(v)
        per[names[-1]] = np.asarray(per_s, dtype=float)
    return corpus, per


def main():
    a = parse()
    cb, pb = score(a.repo, a.reference, a.baseline)
    ca, pa = score(a.repo, a.reference, a.aligned)
    st = pd.read_csv(a.stats)[["id", "status", "valid_area_pct"]]
    big = (st["status"] == "đã căn") & (st["valid_area_pct"] < VALID_MIN)
    st.loc[big, "status"] = "đã căn (lấp viền >=10%)"
    st.loc[st["status"] == "đã căn", "status"] = "đã căn (lấp viền <10%)"

    # Phương án kết hợp: ảnh đã căn cho cặp lấp viền <10%, còn lại giữ câu baseline
    use = set(st.loc[st["status"] == "đã căn (lấp viền <10%)", "id"])
    rows = lambda f: {json.loads(l)["image_id"]: l for l in open(f) if l.strip()}
    rb, ra = rows(a.baseline), rows(a.aligned)
    comb = os.path.splitext(a.out)[0] + "_ket_hop.jsonl"
    with open(comb, "w") as f:
        for k, l in rb.items():
            f.write((ra[k] if k in use and k in ra else l).rstrip("\n") + "\n")
    cc, _ = score(a.repo, a.reference, comb)
    m = pb.merge(pa, on="id", suffixes=("_base", "_align")).merge(st, on="id")
    m.to_csv(a.out, index=False)

    print("\n== Toàn bộ OmniDiff-Real (803 cặp) ==")
    print(f"{'Chỉ số':<9}{'Baseline':>10}{'Căn tất cả':>12}{'Kết hợp':>10}")
    for k in ["Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]:
        print(f"{k:<9}{cb[k]*100:>10.1f}{ca[k]*100:>12.1f}{cc[k]*100:>10.1f}")
    print(f"(Kết hợp = ảnh đã căn cho {len(use)} cặp lấp viền <10%, còn lại ảnh gốc)")

    print("\n== Theo nhóm (điểm trung bình từng câu, thang x100) ==")
    print(f"{'Nhóm':<30}{'n':>5}{'CIDEr base':>12}{'CIDEr căn':>11}{'METEOR base':>13}{'METEOR căn':>12}{'p (Wilcoxon, CIDEr)':>22}")
    for name, g in [("Tất cả", m)] + list(m.groupby("status")):
        diff = g["CIDEr_align"] - g["CIDEr_base"]
        p = wilcoxon(g["CIDEr_align"], g["CIDEr_base"]).pvalue if (diff != 0).sum() >= 10 else float("nan")
        print(f"{name:<30}{len(g):>5}{g['CIDEr_base'].mean()*100:>12.1f}{g['CIDEr_align'].mean()*100:>11.1f}"
              f"{g['METEOR_base'].mean()*100:>13.1f}{g['METEOR_align'].mean()*100:>12.1f}{p:>22.3g}")
    g = m[m["status"].str.startswith("đã căn")]
    if len(g):
        up = int((g["CIDEr_align"] > g["CIDEr_base"]).sum())
        down = int((g["CIDEr_align"] < g["CIDEr_base"]).sum())
        print(f"\nTrong {len(g)} cặp đã căn: {up} cặp điểm tăng, {down} cặp điểm giảm, {len(g)-up-down} cặp không đổi")
    print(f"\nĐã lưu so sánh từng câu: {a.out}")


if __name__ == "__main__":
    main()

## Bước 5c — Căn chỉnh 803 cặp ảnh Real (vài phút, không dùng GPU)

In [ ]:
!{UV} pip install -q --python {PY} opencv-python-headless pandas scipy matplotlib
OUT_ALIGN = f"{VOL}/results/OmniDiff_Real_aligned"
os.makedirs(OUT_ALIGN, exist_ok=True)
Q_ALIGNED = f"{REPO}/data/evaluation/prompt/OmniDiff_Real_test_aligned.json"

!MPLBACKEND=Agg {PY} /root/align_pairs.py --questions {REPO}/data/evaluation/prompt/OmniDiff_Real_test.json \
  --image-root {REPO}/datasets/OmniDiff --out-subdir aligned_Real \
  --out-questions {Q_ALIGNED} --stats {OUT_ALIGN}/align_stats.csv --examples-dir {OUT_ALIGN}/examples

from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{OUT_ALIGN}/examples/*.png")):
    display(Image(p))

Xem ảnh minh hoạ: cột cuối (|A − B| sau khi căn) phải tối hơn hẳn cột kế bên, chỉ còn sáng ở chỗ thay đổi thật.

## Bước 6 — Chạy model trên ảnh đã căn (khoảng 40 phút, chạy tiếp được nếu bị ngắt)

In [ ]:
def count_done(fp):
    if not os.path.exists(fp):
        return 0
    n = 0
    for l in open(fp):
        try:
            json.loads(l); n += 1
        except Exception:
            pass
    return n

ANS_ALIGNED = f"{OUT_ALIGN}/OmniDiff_Real_aligned.jsonl"
print(f"Đã có: {count_done(ANS_ALIGNED)}/803 câu")
if count_done(ANS_ALIGNED) < 803:
    %cd {REPO}
    !PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py \
      --model-path {CKPT} --question-file {Q_ALIGNED} \
      --answers-file {ANS_ALIGNED} --image-folder datasets/OmniDiff --num-chunks 1 --chunk-idx 0 \
      --temperature 0 --torch-dtype {DTYPE} --no-flash-attention
print(f"Hiện có: {count_done(ANS_ALIGNED)}/803 câu")

## Bước 7 — So sánh với baseline

In [ ]:
if shutil.which("java") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y default-jre-headless",
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if shutil.which("java") is None:
    java_home = subprocess.run([PY, "-c", "import jdk; print(jdk.install('11', jre=True, path='/root/.jdk'))"],
                               capture_output=True, text=True).stdout.strip()
    os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

%cd {REPO}
!{PY} /root/compare_runs.py --repo {REPO} --reference data/evaluation/reference/OmniDiff_Real.json \
  --baseline {VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl --aligned {ANS_ALIGNED} \
  --stats {OUT_ALIGN}/align_stats.csv --out {OUT_ALIGN}/compare_per_sample.csv 2>&1 | grep -v "tokenized\|index\|Loading\|DONE\|Done\|testlen\|ratio\|loading"